# Bölüm 5: İlk Python Programınız
**İlk LLM'inizi Oluşturun — Bölüm 5: İlk Python Programınız**

Bu defter, Bölüm 5'teki çalıştırılabilir kod örneklerini içerir. Hücreleri yukarıdan aşağıya doğru çalıştırın.

- Kurulumlar: transformers (GPT-2 demosu için)
- Veri: küçük satır içi metinler; harici dosya gerekmez
- Çalışma zamanı: CPU yeterli; GPU sadece GPT-2 çağrısını hızlandırır

In [ ]:
# ===== KURULUM =====
# transformers kütüphanesini kur (HuggingFace'in LLM'lerle çalışma araç seti)
!pip install -q transformers==4.46.1

import warnings
warnings.filterwarnings('ignore')  # Küçük sürüm uyarılarını sustur

# GPT-2 için HuggingFace araçlarını yükle
# "pipeline" bir model yükleyen ve tüm karmaşıklığı bizim için yöneten bir yardımcıdır
# Bunu önceden oluşturulmuş bir iş akışı olarak düşünün: model yükle → girdiyi işle → çıktı üret
from transformers import pipeline, logging
logging.set_verbosity_error()  # Sadece gerçek hataları göster, bilgi mesajlarını değil

print('Kurulum tamamlandı')

## Hızlı başlangıç: GPT-2 metin üretimi
Bir LLM'yi çalışırken görmek için küçük bir GPT-2 üretimi çalıştırın.

In [ ]:
# GPT-2 metin üretim modelini yükle (124M parametre)
generator = pipeline('text-generation', model='gpt2')

# Başlangıç prompt'undan metin üret
result = generator(
    'The secret to building AI is',  # Başlangıç metni
    max_new_tokens=20,                # 20 kelime daha üret
    do_sample=True,                   # Rastgelelik kullan (sadece en olası kelimeler değil)
    pad_token_id=50256                # Teknik: bir uyarıyı önler
)

# Üretilen metni çıkar ve yazdır
print(result[0]['generated_text'])

## String'ler ve temel işlemler
Metin, uzunluklar ve dilimleme ile çalışma.

**Python ipucu: f-string'ler** değişken değerlerini metne eklemenizi sağlar. String'den önceki `f` "formatted" (biçimlendirilmiş) anlamına gelir ve süslü parantezler `{}` değerlerin nereye ekleneceğini işaretler:
```python
name = "GPT"
print(f"Hello, {name}!")  # → Hello, GPT!
```

In [ ]:
# Bir metin üretim modelinden örnek çıktı
output = 'The secret to building AI is understanding how machines learn from data'
prompt = 'The secret to building AI is'

# Temel string işlemleri
print(len(output))  # Karakter cinsinden uzunluk
print(type(output))  # Bunun bir string olduğunu doğrula

# Kelimelere ayır (string listesi)
words = output.split()
print(words)

# Metot zincirleme: birden fazla işlemi sırayla çağır
# Bu, iki satıra eşdeğerdir:
#   lowercase_text = output.lower()
#   words = lowercase_text.split()
words = output.lower().split()
print(words)

# Sadece üretilen kısmı çıkar (prompt'un sonundan dilimleme)
generated = output[len(prompt):]
print(f'Üretilen: {generated.strip()}')  # .strip() başındaki/sonundaki boşlukları kaldırır
print(f'Kelime sayısı: {len(generated.split())}')

## Sayılar ve biçimlendirme
Temel sayısal değerler ve f-string'ler.

In [ ]:
# Python büyük sayılarda okunabilirlik için alt çizgi kullanır
num_parameters = 124_000_000  # 124 milyon (GPT-2 boyutu)
learning_rate = 0.0001        # Eğitim için küçük adım boyutu
vocab_size = 50257            # GPT-2'nin kelime dağarcığı boyutu

# f-string biçimlendirme püf noktaları:
print(f'GPT-2 {num_parameters:,} parametreye sahip')   # :, virgül ayırıcı ekler
print(f'Öğrenme oranı {learning_rate:.2e}')            # :.2e = bilimsel gösterim
print(f'Kelime dağarcığının yarısı: {vocab_size // 2}')  # {} içinde hesaplama yapabilirsiniz!

## Küçük bir kelime dağarcığı ve tokenizer oluşturma
Oyuncak cümlelerden kelime düzeyinde bir tokenizer'a.

In [ ]:
# Örnek metin verisi (bir modeli eğiteceğiniz şey)
texts = [
    'Yapay zeka oluşturmanın sırrı',
    'Makine öğrenmesinin anahtarı veridir',
    'Yapay zeka sistemleri örneklerden öğrenir'
]

# Tüm metinlerden tüm kelimeleri topla
all_words = []
for text in texts:
    words = text.lower().split()  # Küçük harfe normalleştir
    # extend() her öğeyi ayrı ayrı listeye ekler
    # (append() tüm listeyi TEK öğe olarak eklerdi)
    all_words.extend(words)

print(all_words)

# Liste dilimleme örnekleri
print(all_words[0], all_words[-1], all_words[:3])  # İlk, son, ilk 3

# Kelime dağarcığı oluştur: her benzersiz kelimeyi bir sayıya eşle
vocab = {'<PAD>': 0, '<UNK>': 1}  # Özel token'lar önce (ayrılmış kimlikler)

for word in all_words:
    if word not in vocab:
        # len(vocab) "sonraki kullanılabilir kimliği" verir
        # Vocab 2 öğeye sahipse (kimlikler 0 ve 1), len(vocab)=2 sonraki kimlik olur
        vocab[word] = len(vocab)

print(f'Kelime dağarcığı boyutu: {len(vocab)}')
print(vocab)

# Kelime dağarcığında kelimeleri ara
print(vocab['yapay'], vocab['zeka'])  # Kimliklerini döndürür

## GPT-2 tokenizer ile karşılaştırma
Profesyonel bir tokenizer'ın nasıl farklı olduğunu göster.

**Python ipuçları:**
- `dict.get(key, default)` eğer `key` varsa değeri döndürür, yoksa `default` döndürür. `dict[key]`'den daha güvenlidir çünkü anahtar yoksa çökmez.
- **Liste kavrama (list comprehension)** liste oluşturmanın kompakt bir yoludur. `[expr for item in collection]` listeye ekleme yapan bir for döngüsüne eşdeğerdir.

In [ ]:
# Gerçek GPT-2 tokenizer'ını yükle
from transformers import GPT2Tokenizer
real_tok = GPT2Tokenizer.from_pretrained('gpt2')

# Kelime dağarcığı boyutlarını karşılaştır
print(f'Bizim kelime dağarcığımız: {len(vocab)} kelime')
print(f'GPT-2 kelime dağarcığı: {len(real_tok)} token')

# Bilinmeyen kelime işlemeyi test et
# .get(key, default) anahtar bulunamazsa default döndürür (çökmek yerine)
word = 'sinir'
print(f"'{word}' → {vocab.get(word, vocab['<UNK>'])}")  # <UNK> kimliğini döndürür (1)

# Kelime dağarcığımızla bir cümleyi tokenize et
sentence = 'Sinir ağı hızla öğreniyor'

# Uzun versiyon (açık döngü):
token_ids = []
for word in sentence.lower().split():
    token_id = vocab.get(word, vocab['<UNK>'])  # Kimliği al veya bilinmiyorsa <UNK>
    token_ids.append(token_id)
    print(f'  {word} → {token_id}')

print(f'Token kimlikleri (döngü): {token_ids}')

# Kısa versiyon (liste kavrama - aynı sonuç, daha kompakt):
token_ids = [vocab.get(w, vocab['<UNK>']) for w in sentence.lower().split()]
print(f'Token kimlikleri (kavrama): {token_ids}')

## Tokenize ve detokenize yardımcıları
Bir cümleyi gidiş-dönüş işle.

In [ ]:
# Fonksiyon: Metin → Token Kimlikleri (kodlama)
def tokenize(text, vocab):
    words = text.lower().split()
    return [vocab.get(w, vocab['<UNK>']) for w in words]

# Fonksiyon: Token Kimlikleri → Metin (kod çözme)
def detokenize(ids, vocab):
    # Ters eşleme oluştur (Kimlik → kelime)
    # Sözlük kavrama: {new_key: new_val for key, val in dict.items()}
    # vocab.items() ('yapay', 2), ('zeka', 3), vb. gibi çiftler döndürür
    # Bunları tersine çeviririz: (2, 'yapay'), (3, 'zeka'), vb.
    id_to_word = {v: k for k, v in vocab.items()}
    return ' '.join(id_to_word.get(i, '<UNK>') for i in ids)

# Gidiş-dönüş testi: metin → kimlikler → metin
ids = tokenize('Yapay zeka sırrı', vocab)
print(f'Kodlanmış: {ids}')
print(f'Kod çözülmüş: {detokenize(ids, vocab)}')

# Tokenizer'ımızı GPT-2'ninkiyle karşılaştır
text = 'Yapay zeka sırrı'
print(f'Bizim token\'larımız:   {tokenize(text, vocab)}')
print(f'GPT-2 token\'ları: {real_tok.encode(text)}')  # Farklı! GPT-2 alt kelimeler kullanır, tam kelimeler değil

## Minimal bir tokenizer sınıfı
Durumlu, kelime düzeyinde fit/encode/decode ile tokenizer.

**Python Sınıfları 101:**
Bir **sınıf (class)** veri ve fonksiyonları bir araya getiren nesneler oluşturmak için bir taslaktır.

- `class MyClass:` — taslağı tanımlar
- `__init__(self)` — yeni bir nesne oluşturduğunuzda çalışır (verisini başlatır)
- `self` — "bu belirli nesne"yi ifade eder ("bu araba" vs "genel olarak arabalar" gibi)
- Metotlar (sınıf içindeki fonksiyonlar) otomatik olarak `self`'i ilk parametre olarak alır

**Benzetme:** Bir sınıf araba taslağı gibidir. `__init__` başlangıç özelliklerini kurar (renk, motor boyutu). Taslaktan bir araba ürettiğinizde, `self` O belirli arabayı ifade eder.

In [ ]:
# Nesne yönelimli tokenizer (sınıf veri + metotları bir araya getirir)
class SimpleTokenizer:
    def __init__(self):
        # Özel token'larla kelime dağarcığını başlat
        self.word_to_id = {'<PAD>': 0, '<UNK>': 1}
        self.id_to_word = {0: '<PAD>', 1: '<UNK>'}

    def fit(self, texts):
        """Eğitim metinlerinden kelime dağarcığı oluştur"""
        for text in texts:
            for word in text.lower().split():
                if word not in self.word_to_id:
                    idx = len(self.word_to_id)
                    self.word_to_id[word] = idx  # Yeni kelime ekle
                    self.id_to_word[idx] = word  # Ters eşleme

    def encode(self, text):
        """Metni token kimliklerine dönüştür"""
        return [self.word_to_id.get(w, 1) for w in text.lower().split()]  # 1 = <UNK>

    def decode(self, ids):
        """Token kimliklerini tekrar metne dönüştür"""
        return ' '.join(self.id_to_word.get(i, '<UNK>') for i in ids)

    def __len__(self):
        """Kelime dağarcığı boyutunu döndür (len(tok) kullanımını sağlar)"""
        return len(self.word_to_id)

# Tokenizer oluştur ve eğit
tok = SimpleTokenizer()
tok.fit(texts)  # Eğitim verimizden kelime dağarcığı öğren

print(f'Kelime dağarcığı boyutu: {len(tok)}')

# Kodlama/kod çözmeyi test et
ids = tok.encode('Yapay zeka sırrı')
print(f'Kodlanmış: {ids}')
print(f'Kod çözülmüş: {tok.decode(ids)}')

# GPT-2 ile bir kez daha karşılaştır
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')
text = 'Yapay zeka sırrı'
print(f'Sizin tokenizer\'ınız:  {tok.encode(text)}')
print(f'GPT-2 tokenizer: {gpt2_tok.encode(text)}')  # GPT-2 Byte-Pair Encoding (BPE) kullanır

## Tam Tur: Sizin Tokenizer'ınız vs GPT-2

El yapımı tokenizer'ınızı gerçek GPT-2 tokenizer'ı ile son bir kez karşılaştıralım. GPT-2'nin token kimliklerinin çok daha büyük sayılar olduğuna dikkat edin (50.257 token'a sahip!) ve tam kelimeler yerine **alt kelime tokenizasyonu** (BPE) kullanır.

In [ ]:
# Yeni bir tokenizer oluştur ve GPT-2 ile karşılaştır
my_tok = SimpleTokenizer()
my_tok.fit(['Yapay zeka oluşturmanın sırrı anlayışta'])

# GPT-2'nin tokenizer'ını yükle
from transformers import GPT2Tokenizer
gpt2_tok = GPT2Tokenizer.from_pretrained('gpt2')

# Aynı metin üzerinde karşılaştır
text = 'Yapay zeka sırrı'
print(f'Sizin tokenizer\'ınız:  {my_tok.encode(text)}')
print(f'GPT-2 tokenizer: {gpt2_tok.encode(text)}')

print(f'\nSizin kelime dağarcığı boyutunuz:  {len(my_tok)}')
print(f'GPT-2 kelime dağarcığı boyutu: {len(gpt2_tok)}')

## Ne Oldu?

Sıfırdan eksiksiz bir tokenizer oluşturdunuz! İşte öğrendikleriniz:

1. **String'ler** — `.split()`, `.lower()`, dilimleme ile metin manipülasyonu
2. **Sözlükler** — Kelime dağarcığı araması için anahtar-değer eşlemeleri
3. **Listeler** — Token dizileri için sıralı koleksiyonlar
4. **Fonksiyonlar** — Yeniden kullanılabilir kod blokları (`def tokenize(...)`)
5. **Sınıflar** — Veri + metotları bir araya getiren taslaklar

**Ana fikir:** Tokenizer'ınız tam kelimeler kullanır, bu yüzden bilinmeyen kelimeler `<UNK>` olur. GPT-2, kelimeleri alt kelimelere ayıran **Byte-Pair Encoding (BPE)** kullanır, bu yüzden gerçekten bilinmeyen token'lara nadiren rastlar. Bölüm 8'de bu konuda daha fazla bilgi edineceksiniz!